# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is an mlcroissant.DatasetMetadata object

# Print core metadata information
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets and their fields (all by `@id`).

In [ ]:
from pprint import pprint

# Discover all record sets in the metadata, using their @id
record_sets = metadata.recordSets
if not record_sets:
    print("No record sets found in metadata.")
else:
    print(f"Found {len(record_sets)} record set(s):")
    for idx, rs in enumerate(record_sets):
        print(f"[{idx}] RecordSet @id: {rs['@id']}")
        print(f"    name: {rs.get('name','N/A')}")
        print(f"    description: {rs.get('description','N/A')}")
        if 'field' in rs:
            print("    Fields and their @id:")
            for fie in rs['field']:
                # Field can be a dict or just a string (id)
                # We try to extract the @id and name
                if isinstance(fie, dict):
                    print(f"      - {fie.get('@id')}: {fie.get('name','N/A')}")
                else:
                    print(f"      - {fie}")
        print()

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis. Use the record set and field `@id`s from above.

In [ ]:
# Get list of record set @id's
record_set_ids = [rs['@id'] for rs in metadata.recordSets] if metadata.recordSets else []
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records; columns:", df.columns.tolist())
        print(df.head(2))
    else:
        print("  No records found.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filtering, normalizing numeric fields, grouping. Example uses the first available numeric field.

In [ ]:
# EDA Example: Select first loaded dataframe and try to process a numeric field

import numpy as np

if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    df = dataframes[first_rs_id]

    # Try to find a numeric field by looking at dtypes after conversion
    for col in df.columns:
        with np.errstate(all='ignore'):
            # Try to coerce to float and test for a meaningful result
            try:
                col_numeric = pd.to_numeric(df[col], errors='coerce')
                if col_numeric.notna().sum() > 0 and col != '@id':
                    numeric_field = col
                    break
            except Exception:
                continue
    else:
        numeric_field = None

    if numeric_field:
        print(f"Using field '{numeric_field}' (from {first_rs_id}) as numeric example.")

        # Prepare numeric column
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

        threshold = df[numeric_field].mean()   # Use the mean as illustrative threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.3f} (mean):")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt to group by a likely categorical column (use first string-type column other than @id)
        candidate_group_fields = [c for c in df.columns if df[c].dtype==object and c != '@id' and c != numeric_field]
        if candidate_group_fields:
            group_field = candidate_group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by '{group_field}' and mean '{numeric_field}':")
            print(grouped_df.head())
        else:
            print("No non-numeric grouping field found.")
    else:
        print("No numeric field found for analysis in first record set.")
else:
    print("No dataframes loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Example: histogram and boxplot for the numeric field.

In [ ]:
import matplotlib.pyplot as plt

if dataframes and numeric_field:
    df = dataframes[first_rs_id]
    valid_vals = df[numeric_field].dropna()
    if not valid_vals.empty:
        plt.figure(figsize=(14, 5))
        plt.subplot(1,2,1)
        plt.hist(valid_vals, bins=30, color='steelblue', edgecolor='black')
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")

        plt.subplot(1,2,2)
        plt.boxplot(valid_vals)
        plt.title(f"Boxplot of {numeric_field}")
        plt.xlabel(numeric_field)

        plt.tight_layout()
        plt.show()
    else:
        print(f"Numeric field {numeric_field} is empty or not found.")
else:
    print("No data to plot.")

## 6. Conclusion
This notebook demonstrated how to load and explore a FAIR-enabled dataset defined by a Croissant schema using the `mlcroissant` Python library. We:

- Dynamically loaded the dataset and discovered record sets and their fields by `@id`
- Loaded data from available record sets into DataFrames for analysis
- Performed simple EDA, including filtering, normalization, and grouping
- Visualized a numeric feature's distribution

Further analysis can use more domain-specific logic and additional features. Please make sure to review the metadata and field definitions via their `@id` for precise, reproducible research!